<a href="https://colab.research.google.com/github/speediedan/interpretune/blob/main/src/it_examples/notebooks/publish/quickstart/cpu_quickstart.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

# CPU Quickstart: your first Interpretune analysis in under five minutes

No GPU, no Hugging Face token, no license clicks. This notebook builds a real
`ITSession` on an ungated GPT-2, runs an analysis op over a few RTE examples, and lands the
results in an `AnalysisStore` — the same machinery every larger analysis uses.

Requirements: `uv pip install -e ".[examples]"` (CPU torch is fine).

> **Until the `speediedan/rte` component is public (#261), this notebook needs a Hugging Face token with access to
> it** (set `HF_TOKEN`). After that it runs with no token at all. No GPU and no license acceptance are needed
> either way.

In [2]:
# Parameters - These will be injected by papermill during parameterized test runs
MAX_EXAMPLES = 16
SEED = 0
REVISION = (
    None  # component snapshot; None resolves the recorded pin (pass a local-publish rev to validate unmerged configs)
)

In [4]:
# @title Imports { display-mode: "form" }
import os

import interpretune as it
import interpretune.analysis  # noqa: F401  (registers the it.<op> wrappers)
import torch
from datasets import Dataset

from it_examples.seeds import ensure_local_seeds

torch.manual_seed(SEED)

~/worktrees/interpretune-render/src/interpretune/analysis/ops/dispatcher.py:1001: Alias 'ct_feature_intervention' already has an assigned op or alias so the alias specified by 'feature_intervention_forward' cannot be mapped to it
~/worktrees/interpretune-render/src/interpretune/analysis/ops/dispatcher.py:1001: Alias 'model_forward' already has an assigned op or alias so the alias specified by 'model_fwd' cannot be mapped to it
~/worktrees/interpretune-render/src/interpretune/analysis/ops/dispatcher.py:1001: Alias 'jacobian_lens_read' already has an assigned op or alias so the alias specified by 'jlens_read' cannot be mapped to it
~/worktrees/interpretune-render/src/interpretune/analysis/ops/dispatcher.py:1001: Alias 'jacobian_lens_concept_probe' already has an assigned op or alias so the alias specified by 'jlens_concept_probe' cannot be mapped to it
~/worktrees/interpretune-render/src/interpretune/analysis/ops/dispatcher.py:1001: Alias 'jacobian_lens_sparse_inventory' already has an a

## 1. Session construction

Hub-first: fetch the `speediedan/rte` component, load its `rte_demo.gpt2.core` config (CPU-only, ungated
model), compose a core session, initialize it.

Loading a component runs the Python code it publishes (here, the RTE task module), so Interpretune refuses until
you opt in. `it.hub.pull` caches the repository **without executing anything**, so you can read the entrypoint
first; the opt-in then applies for this session. Pin the revision you trusted if you reuse this outside a demo.

In [5]:
# @title 1: Session construction { display-mode: "form" }
ensure_local_seeds()  # idempotent, offline: seed publish sources -> local components cache
if REVISION is None:  # a validation run passes a local-publish REVISION and needs neither step
    it.hub.pull("speediedan/rte")  # caches the component; executes nothing
    os.environ["IT_TRUST_REMOTE_CODE"] = "1"  # opt in to running its entrypoint, for this session only
base_itdm_cfg, base_it_cfg, dm_cls, m_cls = it.hub.load("speediedan/rte", "rte_demo.gpt2.core", revision=REVISION)

session_cfg = it.ITSessionConfig(
    adapter_ctx=(it.Adapter.core,),
    datamodule_cfg=base_itdm_cfg,
    module_cfg=base_it_cfg,
    datamodule_cls=dm_cls,
    module_cls=m_cls,
)
it_session = it.ITSession(session_cfg)
it.it_init(it_session.module, it_session.datamodule)
print("session ready:", type(it_session.module).__name__)

~/worktrees/interpretune-render/src/interpretune/hub/components.py:480: 'speediedan/rte' resolved to the local-publish snapshot locale51bcbd, which shadows the cached Hub revision(s) ['738e41229e39', '8f61b54c4d2c', 'a9b2407a96e5']; the Hub was not consulted (resolution is cache-only). If you meant the published artifact, run interpretune.hub.pull('speediedan/rte') or resolve with require_hub=True.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'GPT2Config' object has no attribute 'quantization_config'


[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'GPT2Config' object has no attribute '_pre_quantization_dtype'


[INFO] interpretune.utils.logging: Preparing data: InterpretunableDataModule


...

Saving the dataset (0/1 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

[INFO] interpretune.utils.logging: Setting up datamodule: InterpretunableDataModule


[INFO] interpretune.utils.logging: Setting up model: InterpretunableModule


[INFO] interpretune.utils.logging: initializing optimizers and schedulers: InterpretunableModule


session ready: InterpretunableModule


## 2. Run analysis

One op at a time over a small slice of the RTE validation split, collecting predictions. Each
call below is the public op surface — the same calls a composed pipeline makes.

In [6]:
# @title 2: Run the analysis { display-mode: "form" }
from interpretune import AnalysisCfg
from interpretune.analysis.ops.base import AnalysisBatch, get_batch_input
from interpretune.runners.analysis import maybe_init_analysis_cfg
from transformers import BatchEncoding

module = it_session.module
seen, preds_all, labels_all, diffs_all = 0, [], [], []
for batch in module.datamodule.test_dataloader():
    if seen >= MAX_EXAMPLES:
        break
    n = get_batch_input(batch).size(0)
    take = min(n, MAX_EXAMPLES - seen)
    # Stay a BatchEncoding: the ops resolve the model's input key from it
    slim = BatchEncoding({k: (v[:take] if hasattr(v, "__getitem__") else v) for k, v in batch.items()})
    ab = AnalysisBatch()
    for op in (it.labels_to_ids, it.model_fwd, it.logit_diffs):
        maybe_init_analysis_cfg(module, AnalysisCfg(target_op=op, ignore_manual=True))
        ab = op(module, ab, slim, 0)
    preds_all.append(ab.preds.detach().cpu())
    labels_all.append(ab.orig_labels.detach().cpu())
    diffs_all.append(ab.logit_diffs.detach().cpu())
    seen += take

preds = torch.cat(preds_all)
labels = torch.cat(labels_all)
print(f"ran {seen} examples")

ran 16 examples


## 3. Interpret: accuracy on the slice, landed in an AnalysisStore

In [7]:
# @title 3: Interpret { display-mode: "form" }
from interpretune.analysis.core import AnalysisStore

accuracy = (preds == labels).float().mean().item()
print(f"accuracy on {seen} RTE examples: {accuracy:.3f}")
print(f"mean logit diff (correct - incorrect): {torch.cat(diffs_all).float().mean().item():+.3f}")

store = AnalysisStore(dataset=Dataset.from_dict({"preds": preds.tolist(), "labels": labels.tolist()}))
print("store rows:", store.dataset.num_rows)

accuracy on 16 RTE examples: 0.438
mean logit diff (correct - incorrect): -0.013
store rows: 16


## Where next

- The [examples gallery](https://interpretune.org/en/latest/examples.html) runs the same machinery
  with SAE latents, ablations and attributions (those need a GPU and, for Gemma, gated weights).
- The module-construction tour (session shapes, composition, MRO) is a separate notebook built on
  this config's substrate.
